In [2]:
"""
Sat Mar 14 17:04:29 CST 2026
author: pengchen

"""
import sys
sys.path.append("../src")
from utils import LevelTriggeredDTypeFlipFlopWithClear, Inverter, AND, bit_list_to_int, OR

class NBitsMemory:
    """
    one time store nbits data, not flexible but simple to implement.
    """
    def __init__(self, n_bits):
        self.n_bits = n_bits
        self.latches = [LevelTriggeredDTypeFlipFlopWithClear() for _ in range(n_bits)]
    
    def __call__(self, data, write_enable):
        assert len(data) == self.n_bits, f"Data bus must be {self.n_bits} bits long"
        assert write_enable in [0, 1], f"Write enable must be 0 or 1"
        # data[-1] is the LSB, data[0] is the MSB
        for i in reversed(range(self.n_bits)):
            self.latches[i]([data[i], write_enable])  # D = data[i], CLK = write_enable
        return self.read()

    def read(self):
        bits = [bit.getQ() for bit in self.latches]
        return bits # return MSB first, LSB last
    
class Decoder_3_8:
    """
    3 to 8 decoder, takes 3 input bits and decodes them into 8 output lines.
    Only one output line will be high (1) at a time
    for example, if the input is 101, then the output will be 00100000
    """
    def __init__(self, nin = 3, nout = 8):
        self.nin = nin
        self.nout = nout
        self.inverters = [Inverter() for _ in range(self.nin)]
        self.and_gates = [AND(4) for _ in range(self.nout)]
    
    def __call__(self, address, write):
        assert len(address) == self.nin, "Input must be 3 bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"

        address = [[address[i]] for i in range(len(address))] # convert to list of lists for inverters gate input

        output = [0] * self.nout
        output[7] = self.and_gates[0]([write, self.inverters[2](address[2]), self.inverters[1](address[1]), self.inverters[0](address[0])]) # 1000
        output[6] = self.and_gates[1]([write, self.inverters[2](address[2]), self.inverters[1](address[1]), address[0][0]]) # 1001
        output[5] = self.and_gates[2]([write, self.inverters[2](address[2]), address[1][0], self.inverters[0](address[0])]) # 1010
        output[4] = self.and_gates[3]([write, self.inverters[2](address[2]), address[1][0], address[0][0]]) # 1011
        output[3] = self.and_gates[4]([write, address[2][0], self.inverters[1](address[1]), self.inverters[0](address[0])]) # 1100
        output[2] = self.and_gates[5]([write, address[2][0], self.inverters[1](address[1]), address[0][0]]) # 1101
        output[1] = self.and_gates[6]([write, address[2][0], address[1][0], self.inverters[0](address[0])]) # 1110
        output[0] = self.and_gates[7]([write, address[2][0], address[1][0], address[0][0]]) # 1111
        # output[0] is MSB
        return output

class Selector_8_1:
    """
    8 to 1 selector, takes 8 input lines and selects one of them based on a 3-bit address.
    for example, if the address is 101, then the output will be the value of the 00100000 -> output[2] out_from_memory line
    """
    def __init__(self, nin = 3, nout = 8):
        self.nin = nin
        self.nout = nout
        self.inverters = [Inverter() for _ in range(self.nin)]
        self.and_gates = [AND(4) for _ in range(self.nout)]
        self.or_gate = OR(self.nout)

    def __call__(self, address, out_from_memory):
        assert len(address) == self.nin, f"Input must be {self.nin} bits long"
        assert len(out_from_memory) == self.nout, f"Out_from_memory must be {self.out} bits long"
        address = [[address[i]] for i in range(len(address))] # convert to list of lists for inverters gate input

        output = [0] * self.nout
        output[7] = self.and_gates[0]([out_from_memory[7], self.inverters[2](address[2]), self.inverters[1](address[1]), self.inverters[0](address[0])]) # 1000
        output[6] = self.and_gates[1]([out_from_memory[6], self.inverters[2](address[2]), self.inverters[1](address[1]), address[0][0]]) # 1001
        output[5] = self.and_gates[2]([out_from_memory[5], self.inverters[2](address[2]), address[1][0], self.inverters[0](address[0])]) # 1010
        output[4] = self.and_gates[3]([out_from_memory[4], self.inverters[2](address[2]), address[1][0], address[0][0]]) # 1011
        output[3] = self.and_gates[4]([out_from_memory[3], address[2][0], self.inverters[1](address[1]), self.inverters[0](address[0])]) # 1100
        output[2] = self.and_gates[5]([out_from_memory[2], address[2][0], self.inverters[1](address[1]), address[0][0]]) # 1101
        output[1] = self.and_gates[6]([out_from_memory[1], address[2][0], address[1][0], self.inverters[0](address[0])]) # 1110
        output[0] = self.and_gates[7]([out_from_memory[0], address[2][0], address[1][0], address[0][0]]) # 1111
        out = self.or_gate(output)
        return [out]
    

class RAM_8_1:
    """
    A simple memory module that combines the decoder, memory cells, and selector to create a functional memory unit.
    For simplicity, we will implement a 8-word memory with 3-bit addresses and 1-bit data.
    """
    def __init__(self):
        self.capacity = 8  # 8 words
        self.word_size = 1  # 1 bit per word

        self.decoder = Decoder_3_8()
        self.memory_cells = [NBitsMemory(self.word_size) for _ in range(self.capacity)]  # 8 words of 1 bit each
        self.selector = Selector_8_1()

    def __call__(self, address, data_in, write):
        assert len(address) == 3, "Address must be 3 bits long"
        assert len(data_in) == self.word_size, f"Data input must be {self.word_size} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"
        
        # Use the decoder to determine which memory cell to write to
        write_lines = self.decoder(address, write)
        for i in range(self.capacity):
            self.memory_cells[i](data_in, write_lines[i])  # Write data to the selected memory cell
        return self.read(address)

    def read(self, address):
        # Read from all memory cells and use the selector to get the output from the selected cell
        out_from_memory = [cell.read()[0] for cell in self.memory_cells]  # Get the output from all memory cells
        return self.selector(address, out_from_memory)  # Select the output from the addressed cell


In [3]:
def test_memory():
    """Proves the decoder NEVER lights up more than one wire at a time."""
    decoder = Decoder_3_8()

    # Test address 101 (Decimal 5) with Write Enable = 1
    # Because of your reverse indexing, 101 should light up index 2
    out = decoder([1, 0, 1], 1)

    assert sum(out) == 1, "FATAL: Decoder lit up multiple wires! Short circuit!"
    assert out[2] == 1, "Decoder sent the signal to the wrong memory bank."

    # Test Write Enable Shielding
    out_disabled = decoder([1, 0, 1], 0)
    assert sum(out_disabled) == 0, "FATAL: Decoder leaked signal when Write Enable was 0!"

    """Proves memory ignores the data bus when Write Enable is 0."""
    ram = RAM_8_1()
    address = [0, 1, 1] # Address 3

    # Write a 1 to Address 3
    ram(address, [1], write=1)
    assert ram.read(address) == [1], "Failed to write 1."

    # Now, put a 0 on the data bus, but set Write Enable = 0
    ram(address, [0], write=0)

    # Read it back. It should still be 1!
    assert ram.read(address) == [1], "GHOST WRITE: Memory overwrote data while WE=0!"

    """Writes a unique pattern to all 8 cells, then verifies none of them bled into each other."""
    ram = RAM_8_1()

    # 3-bit addresses from 000 to 111
    addresses = [
        [0,0,0], [0,0,1], [0,1,0], [0,1,1],
        [1,0,0], [1,0,1], [1,1,0], [1,1,1]
    ]

    # We will write the pattern: 0, 1, 0, 1, 0, 1, 0, 1
    for i, addr in enumerate(addresses):
        data_bit = i % 2
        ram(addr, [data_bit], write=1)
        
    # Now verify the entire array. If ANY address bled, this will fail.
    for i, addr in enumerate(addresses):
        expected_bit = i % 2
        actual_bit = ram.read(addr)[0]
        assert actual_bit == expected_bit, f"CROSSTALK at address {addr}. Expected {expected_bit}, got {actual_bit}"

    """Slams a single memory cell with alternating data to ensure latches don't get stuck."""
    ram = RAM_8_1()
    addr = [1, 1, 1]

    for i in range(50):
        # Alternate between writing 0 and 1
        val = i % 2
        ram(addr, [val], write=1)
        
        # Read it immediately
        assert ram.read(addr) == [val], f"Latch stuck! Failed on write cycle {i}"


In [4]:
class RAM_8_8:
    """
    A simple memory module that combines the decoder, memory cells, and selector to create a functional memory unit.
    For simplicity, we will implement a 8-word memory with 3-bit addresses and 8-bit data.
    """
    def __init__(self, word_size=8):
        self.capacity = 8  # 8 words
        self.word_size = word_size
        self.ram_8_1_cells = [RAM_8_1() for _ in range(self.word_size)]  # 8 words of 8 bits each
        
    def __call__(self, address, data_in, write):
        assert len(address) == 3, "Address must be 3 bits long"
        assert len(data_in) == self.word_size, f"Data input must be {self.word_size} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"

        for bit in range(self.word_size):
            self.ram_8_1_cells[bit](address, [data_in[bit]], write)  # Write each bit to the selected memory cell
        return self.read(address)

    def read(self, address):
        bits = []
        for bit in range(self.word_size):
            bits = bits + self.ram_8_1_cells[bit].read(address)
        return bits

In [5]:
def test_ram8x8():
    # ==========================================
    # 1. BASIC BYTE READ/WRITE
    # ==========================================
    # def test_ram8x8_basic_rw():
        """Tests if a full 8-bit byte can be written and retrieved."""
        ram = RAM_8_8()
        addr = [0, 1, 0] # Address 2
        data = [1, 1, 0, 0, 1, 0, 1, 0]
        
        # Write the byte
        ram(addr, data, write=1)
        
        # Read the byte back
        out = ram.read(addr)
        assert out == data, f"Basic read/write failed. Expected {data}, got {out}"

    # ==========================================
    # 2. THE GHOST WRITE (WRITE ENABLE SHIELD)
    # ==========================================
    # def test_ram8x8_ghost_writes():
        """Proves the 8-bit data bus is ignored when Write Enable is 0."""
        ram = RAM_8_8()
        addr = [1, 1, 1] # Address 7
        initial_data = [0, 0, 0, 0, 1, 1, 1, 1]
        
        # 1. Properly write the initial data
        ram(addr, initial_data, write=1)
        
        # 2. Flood the data bus with a completely new byte, but keep WE=0
        ghost_data = [1, 1, 1, 1, 0, 0, 0, 0]
        ram(addr, ghost_data, write=0)
        
        # 3. Read it back. It should NOT be the ghost data!
        out = ram.read(addr)
        assert out == initial_data, f"Memory corrupted by ghost write! WE=0 was ignored."

    # ==========================================
    # 3. MASSIVE CROSSTALK STRESS TEST
    # ==========================================
    # def test_ram8x8_address_crosstalk():
        """
        Fills all 8 bytes of memory with completely unique, alternating bit patterns.
        Then reads all 8 bytes back to ensure no rows or columns bled into each other.
        """
        ram = RAM_8_8()
        
        # All 8 possible 3-bit addresses
        addresses = [
            [0,0,0], [0,0,1], [0,1,0], [0,1,1],
            [1,0,0], [1,0,1], [1,1,0], [1,1,1]
        ]
        
        # 8 highly distinct 8-bit patterns
        patterns = [
            [0,0,0,0,0,0,0,0], # All low
            [1,1,1,1,1,1,1,1], # All high
            [1,0,1,0,1,0,1,0], # Alternating A
            [0,1,0,1,0,1,0,1], # Alternating B
            [1,1,0,0,1,1,0,0], # Pairs A
            [0,0,1,1,0,0,1,1], # Pairs B
            [1,1,1,1,0,0,0,0], # Half-n-half A
            [0,0,0,0,1,1,1,1]  # Half-n-half B
        ]
        
        # Step 1: Write all patterns sequentially
        for addr, data in zip(addresses, patterns):
            ram(addr, data, write=1)
            
        # Step 2: Verify the entire memory array is intact
        for addr, expected_data in zip(addresses, patterns):
            out = ram.read(addr)
            assert out == expected_data, f"Crosstalk at address {addr}! Expected {expected_data}, got {out}"

    # ==========================================
    # 4. BYTE REWRITE FATIGUE
    # ==========================================
    # def test_ram8x8_rewrite_fatigue():
        """Rapidly flips every single bit in a specific byte to ensure latches don't lock up."""
        ram = RAM_8_8()
        addr = [1, 0, 1] # Address 5
        
        pattern1 = [1, 0, 1, 0, 1, 0, 1, 0]
        pattern2 = [0, 1, 0, 1, 0, 1, 0, 1]
        
        for i in range(20):
            # Swap the pattern on every iteration
            data = pattern1 if i % 2 == 0 else pattern2
            
            # Write and instantly verify
            ram(addr, data, write=1)
            out = ram.read(addr)
            assert out == data, f"Fatigue failure on write cycle {i}. Bits got stuck!"
test_ram8x8()

In [6]:
class Decoder_4_16:
    """
    4 to 16 decoder, takes 4 input bits and decodes them into 16 output lines.
    Only one output line will be high (1) at a time
    for example, if the input is 1010, then the output will be 0000010000000000
    """
    def __init__(self, nin = 4, nout = 16):
        self.nin = nin
        self.nout = nout
        self.inverters = [Inverter() for _ in range(self.nin)]
        self.and_gates = [AND(4) for _ in range(self.nout)]
        self.and_gates_with_write = [AND(2) for _ in range(self.nout)]
    
    def write(self, inputs, write):
        assert len(inputs) == self.nout, f"Inputs must be {self.nout} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"
        # This function will take the write signal and the inputs, and return the output of the AND gate
        outputs = []
        for i in range(self.nout):
            outputs.append(self.and_gates_with_write[i]([write, inputs[i]]))
        return outputs

    def __call__(self, address):
        assert len(address) == self.nin, "Input must be 4 bits long"
        idxs = [
            [0, 0, 0, 0], 
            [0, 0, 0, 1], 
            [0, 0, 1, 0], 
            [0, 0, 1, 1], 
            [0, 1, 0, 0], 
            [0, 1, 0, 1], 
            [0, 1, 1, 0], 
            [0, 1, 1, 1], 
            [1, 0, 0, 0], 
            [1, 0, 0, 1], 
            [1, 0, 1, 0], 
            [1, 0, 1, 1], 
            [1, 1, 0, 0], 
            [1, 1, 0, 1], 
            [1, 1, 1, 0], 
            [1, 1, 1, 1]
        ]
        address = [[address[i]] for i in range(len(address))] # convert to list of lists for inverters gate input
        output = [0] * self.nout
        # like Decoder_3_8, but just use loop instead of hardcoding each line
        for i, idx in enumerate(idxs):
            # print(f"Address: {idx} -> Output Line: {15 - i}")
            input_and = []
            for j, val in enumerate(idx):
                if val == 0:
                    input_and.append(self.inverters[j](address[j]))
                else:
                    input_and.append(address[j][0])
            # output[self.nout - 1 - i] = self.and_gates[i]([write] + input_and)  # write is the first input to the AND gate
            output[self.nout - 1 - i] = self.and_gates[i](input_and)
        # output[0] is MSB
        return output

class Selector_16_1:
    """
    16 to 1 selector, takes 16 input lines and selects one of them based on a 16-bit address.
    for example, if the address is 1010, then the output will be the value of the 0000010000000000 -> output[5] out_from_memory line
    """
    def __init__(self, nin = 4, nout = 16):
        self.nin = nin
        self.nout = nout
        self.inverters = [Inverter() for _ in range(self.nin)]
        self.and_gates = [AND(5) for _ in range(self.nout)]
        self.or_gate = OR(self.nout)

    def __call__(self, address, out_from_memory):
        assert len(address) == self.nin, f"Input must be {self.nin} bits long"
        assert len(out_from_memory) == self.nout, f"Out_from_memory must be {self.nout} bits long"
        idxs = [
            [0, 0, 0, 0], 
            [0, 0, 0, 1], 
            [0, 0, 1, 0], 
            [0, 0, 1, 1], 
            [0, 1, 0, 0], 
            [0, 1, 0, 1], 
            [0, 1, 1, 0], 
            [0, 1, 1, 1], 
            [1, 0, 0, 0], 
            [1, 0, 0, 1], 
            [1, 0, 1, 0], 
            [1, 0, 1, 1], 
            [1, 1, 0, 0], 
            [1, 1, 0, 1], 
            [1, 1, 1, 0], 
            [1, 1, 1, 1]
        ]
        address = [[address[i]] for i in range(len(address))] # convert to list of lists for inverters gate input
        output = [0] * self.nout
        # like Decoder_3_8, but just use loop instead of hardcoding each line
        for i, idx in enumerate(idxs):
            # print(f"Address: {idx} -> Output Line: {15 - i}")
            input_and = []
            for j, val in enumerate(idx):
                if val == 0:
                    input_and.append(self.inverters[j](address[j]))
                else:
                    input_and.append(address[j][0])
            output[self.nout - 1 - i] = self.and_gates[i]([out_from_memory[self.nout - 1 - i]] + input_and)  # write is the first input to the AND gate
        out = self.or_gate(output)
        return [out]
    

class RAM_16_1:
    """
    A simple memory module that combines the decoder, memory cells, and selector to create a functional memory unit.
    For simplicity, we will implement a 16-word memory with 4-bit addresses and 1-bit data.
    """
    def __init__(self):
        self.capacity = 16  # 8 words
        self.word_size = 1  # 1 bit per word

        self.decoder = Decoder_4_16()
        self.memory_cells = [NBitsMemory(self.word_size) for _ in range(self.capacity)]  # 8 words of 1 bit each
        self.selector = Selector_16_1()

    def __call__(self, address, data_in, write):
        assert len(address) == 4, "Address must be 4 bits long"
        assert len(data_in) == self.word_size, f"Data input must be {self.word_size} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"
        
        # Use the decoder to determine which memory cell to write to
        write_lines = self.decoder(address)
        write_lines = self.decoder.write(write_lines, write)

        for i in range(self.capacity):
            self.memory_cells[i](data_in, write_lines[i])  # Write data to the selected memory cell
        return self.read(address)

    def read(self, address):
        # Read from all memory cells and use the selector to get the output from the selected cell
        out_from_memory = [cell.read()[0] for cell in self.memory_cells]  # Get the output from all memory cells
        return self.selector(address, out_from_memory)  # Select the output from the addressed cell

class RAM_16_8:
    """
    A simple memory module that combines the decoder, memory cells, and selector to create a functional memory unit.
    For simplicity, we will implement a 16-word memory with 4-bit addresses and 8-bit data.
    """
    def __init__(self, word_size=8):
        self.capacity = 16  # 16 words
        self.word_size = word_size
        self.ram_16_1_cells = [RAM_16_1() for _ in range(self.word_size)]  # 16 words of 8 bits each
        
    def __call__(self, address, data_in, write):
        assert len(address) == 4, "Address must be 4 bits long"
        assert len(data_in) == self.word_size, f"Data input must be {self.word_size} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"

        for bit in range(self.word_size):
            self.ram_16_1_cells[bit](address, [data_in[bit]], write)  # Write each bit to the selected memory cell
        return self.read(address)

    def read(self, address):
        bits = []
        for bit in range(self.word_size):
            bits = bits + self.ram_16_1_cells[bit].read(address)
        return bits



In [7]:
# %%writefile test_07_ram16x8.py

# import sys
# sys.path.append("../src")
# from utils import RAM_16_8, Decoder_4_16, Selector_16_1

def test_ram16_8():
# ==========================================
# 1. DECODER 4-to-16 ISOLATION TEST
# ==========================================
# def test_decoder_4_16_isolation():
    """Proves the 16-channel decoder NEVER lights up multiple wires."""
    decoder = Decoder_4_16()
    
    # Test address 1010 (Decimal 10)
    out = decoder([1, 0, 1, 0])
    out = decoder.write(out, write=1)
    assert sum(out) == 1, "FATAL: Decoder lit up multiple wires! Short circuit!"
    # Because of your reversed mapping, index 5 should be lit (15 - 10 = 5)
    assert out[5] == 1, "Decoder routed to the wrong memory bank."
    
    # Test Write Enable Shield
    out_disabled = decoder([1, 0, 1, 0])
    out_disabled = decoder.write(out_disabled, write=0)
    assert sum(out_disabled) == 0, "FATAL: Decoder leaked signal when WE=0!"

# ==========================================
# 2. THE GHOST WRITE TEST
# ==========================================
# def test_ram16x8_ghost_writes():
    """Proves the 16x8 memory ignores the 8-bit data bus when WE=0."""
    ram = RAM_16_8()
    addr = [1, 1, 1, 1] # Address 15
    initial_data = [1, 0, 1, 0, 1, 0, 1, 0]
    
    # Write the data
    ram(addr, initial_data, write=1)
    
    # Flood data bus with 1s, but WE=0
    ram(addr, [1, 1, 1, 1, 1, 1, 1, 1], write=0)
    
    # Verify it was blocked!
    out = ram.read(addr)
    assert out == initial_data, "GHOST WRITE: Memory corrupted while WE=0."

# ==========================================
# 3. MASSIVE 16-BYTE CROSSTALK STRESS TEST
# ==========================================
# def test_ram16x8_massive_crosstalk():
    """
    Writes 16 completely unique byte patterns to all 16 memory addresses.
    Then reads them all back to ensure absolute isolation across all 128 bits.
    """
    ram = RAM_16_8()
    
    # Generate all 16 addresses dynamically (0000 to 1111)
    addresses = [[(i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1] for i in range(16)]
    # print(addresses)
    
    # Generate 16 completely unique 8-bit patterns
    # We will just use the binary representation of numbers 20 through 35
    patterns = [[(val >> j) & 1 for j in range(7, -1, -1)] for val in range(20, 36)]
    # print(patterns)
    # 1. Write Phase: Slam the RAM with all 16 patterns
    for addr, data in zip(addresses, patterns):
        ram(addr, data, write=1)
        
    # 2. Read Phase: Verify no row overwrote any other row
    for addr, expected_data in zip(addresses, patterns):
        out = ram.read(addr)
        assert out == expected_data, f"CROSSTALK BLEED at address {addr}! Expected {expected_data}, got {out}"

# ==========================================
# 4. EXTREME REWRITE FATIGUE
# ==========================================
# def test_ram16x8_rewrite_fatigue():
    """Rapidly writes to the exact same cell 50 times to ensure latches clear correctly."""
    ram = RAM_16_8()
    addr = [0, 1, 0, 1] # Address 5
    
    pattern1 = [1, 1, 1, 1, 0, 0, 0, 0]
    pattern2 = [0, 0, 0, 0, 1, 1, 1, 1]
    
    for i in range(50):
        data = pattern1 if i % 2 == 0 else pattern2
        ram(addr, data, write=1)
        out = ram.read(addr)
        assert out == data, f"Fatigue failure on cycle {i}. Latches failed to update!"
test_ram16_8()

In [8]:
class RAM_64KB:
    """
    64KB Memory using 16-bit address bus.
    Address are split:
        - Top 12 bits route to one of 4,096 RAM_16_8 chips.
        - Bottom 4 bits select the exact byte within that chip.
    """
    def __init__(self, word_size=8):
        self.capacity = 65536  # 64KB = 65536 bytes
        self.word_size = word_size
        self.ram_16_8_cells = [RAM_16_8() for _ in range(self.capacity // 16)]  # Each RAM_16_8 can handle 16 addresses
        self.decoder0 = Decoder_4_16()  # To select which RAM_16_8 to use based on the address[0:4]
        self.decoder1 = Decoder_4_16()  # To select which RAM_16_8 to use based on the address[4:8]
        self.decoder2 = Decoder_4_16()  # To select which RAM_16_8 to use based on the address[8:12]

    def __call__(self, address, data_in, write):
        assert len(address) == 16, "Address must be 16 bits long"
        assert len(data_in) == self.word_size, f"Data input must be {self.word_size} bits long"
        assert write in [0, 1], "Write signal must be 0 or 1"

        # 1. Decode the top 12 bits to find the active RAM_16_8 chip
        out0 = self.decoder0(address[:4])
        out1 = self.decoder1(address[4:8])
        out2 = self.decoder2(address[8:12])
        for i, val0 in enumerate(out0):
            if val0 == 0: continue # software magic: skip dead wires to save time.
            for j, val1 in enumerate(out1):
                if val1 == 0: continue
                for k, val2 in enumerate(out2):
                    if val2 == 0: continue
                    # if val0, val1, and val2 are all 1, we found the active chip
                    idx = (i << 8) | (j << 4) | k
                    # the write signal is passed directly into the smaller chip1
                    self.ram_16_8_cells[idx](address[12:], data_in, write)
        
        return self.read(address)

    def read(self, address):
        out0 = self.decoder0(address[:4])
        out1 = self.decoder1(address[4:8])
        out2 = self.decoder2(address[8:12])
        for i, val0 in enumerate(out0):
            if val0 == 0: continue # software magic: skip dead wires to save time.
            for j, val1 in enumerate(out1):
                if val1 == 0: continue
                for k, val2 in enumerate(out2):
                    if val2 == 0: continue
                    """
                    page 281 of the book, read need [enable] pin, but 
                    here we needn't, because we use if condition to check
                    idx is our active chip.
                    """
                    # if val0, val1, and val2 are all 1, we found the active chip
                    idx = (i << 8) | (j << 4) | k
                    # the write signal is passed directly into the smaller chip1
                    outputs = self.ram_16_8_cells[idx].read(address[12:])
        return outputs

In [11]:
# %%writefile test_08_ram64kb.py

# import sys
# sys.path.append("../src")
# from utils import RAM_64KB

def test_ram64kb():
# ==========================================
# 1. EXTREME BOUNDARIES TEST
# ==========================================
# def test_64kb_extreme_boundaries():
    """Tests writing to the absolute edges of the 64KB memory space."""
    ram = RAM_64KB()
    
    # Address 0: 0000 0000 0000 0000
    addr_first = [0]*16
    data_first = [1, 0, 1, 0, 1, 0, 1, 0]
    
    # Address 65535: 1111 1111 1111 1111
    addr_last = [1]*16
    data_last = [0, 1, 0, 1, 0, 1, 0, 1]
    
    # Write to opposite ends of the universe
    ram(addr_first, data_first, write=1)
    ram(addr_last, data_last, write=1)
    
    # Verify both stuck
    assert ram.read(addr_first) == data_first, "Failed to read Address 0x0000"
    assert ram.read(addr_last) == data_last, "Failed to read Address 0xFFFF"

# ==========================================
# 2. CHIP SELECT CROSSTALK STRESS TEST
# ==========================================
# def test_64kb_chip_crosstalk():
    """
    Writes to 3 different RAM_16_8 chips and verifies the routing logic
    didn't accidentally wake up the wrong chips on the motherboard.
    """
    # ram = RAM_64KB()
    
    # # Address A: Chip 0, internal offset 5
    addr_A = [0,0,0,0, 0,0,0,0, 0,0,0,0, 0,1,0,1]
    data_A = [1, 1, 0, 0, 0, 0, 1, 1]
    
    # # Address B: Chip 1, internal offset 5
    addr_B = [0,0,0,0, 0,0,0,0, 0,0,0,1, 0,1,0,1]
    data_B = [0, 0, 1, 1, 1, 1, 0, 0]
    
    # # Address C: Chip 4095, internal offset 5
    addr_C = [1,1,1,1, 1,1,1,1, 1,1,1,1, 0,1,0,1]
    data_C = [1, 1, 1, 1, 1, 1, 1, 1]
    
    # # Write
    ram(addr_A, data_A, write=1)
    ram(addr_B, data_B, write=1)
    ram(addr_C, data_C, write=1)
    
    # # Read back
    assert ram.read(addr_A) == data_A, "Crosstalk corrupted Chip 0"
    assert ram.read(addr_B) == data_B, "Crosstalk corrupted Chip 1"
    assert ram.read(addr_C) == data_C, "Crosstalk corrupted Chip 4095"

# ==========================================
# 3. GHOST WRITE TEST (64KB SCALE)
# ==========================================
# def test_64kb_ghost_write():
    # ram = RAM_64KB()
    addr = [0,1,0,1, 0,1,0,1, 0,1,0,1, 0,1,0,1] # 0x5555
    data_valid = [0, 0, 0, 0, 1, 1, 1, 1]
    data_ghost = [1, 1, 1, 1, 0, 0, 0, 0]
    
    # # Valid write
    ram(addr, data_valid, write=1)
    
    # # Ghost write (Write Enable = 0)
    ram(addr, data_ghost, write=0)
    
    assert ram.read(addr) == data_valid, "WE=0 shield failed at 64KB scale!"

test_ram64kb()

In [8]:
idxs = [
    [0, 0, 0, 0], 
    [0, 0, 0, 1], 
    [0, 0, 1, 0], 
    [0, 0, 1, 1], 
    [0, 1, 0, 0], 
    [0, 1, 0, 1], 
    [0, 1, 1, 0], 
    [0, 1, 1, 1], 
    [1, 0, 0, 0], 
    [1, 0, 0, 1], 
    [1, 0, 1, 0], 
    [1, 0, 1, 1], 
    [1, 1, 0, 0], 
    [1, 1, 0, 1], 
    [1, 1, 1, 0], 
    [1, 1, 1, 1]
]
for i, idx in enumerate(idxs):
    print(f"Address: {idx} -> Output Line: {15 - i}")
    for j in idx:
        print(j)  

Address: [0, 0, 0, 0] -> Output Line: 15
0
0
0
0
Address: [0, 0, 0, 1] -> Output Line: 14
0
0
0
1
Address: [0, 0, 1, 0] -> Output Line: 13
0
0
1
0
Address: [0, 0, 1, 1] -> Output Line: 12
0
0
1
1
Address: [0, 1, 0, 0] -> Output Line: 11
0
1
0
0
Address: [0, 1, 0, 1] -> Output Line: 10
0
1
0
1
Address: [0, 1, 1, 0] -> Output Line: 9
0
1
1
0
Address: [0, 1, 1, 1] -> Output Line: 8
0
1
1
1
Address: [1, 0, 0, 0] -> Output Line: 7
1
0
0
0
Address: [1, 0, 0, 1] -> Output Line: 6
1
0
0
1
Address: [1, 0, 1, 0] -> Output Line: 5
1
0
1
0
Address: [1, 0, 1, 1] -> Output Line: 4
1
0
1
1
Address: [1, 1, 0, 0] -> Output Line: 3
1
1
0
0
Address: [1, 1, 0, 1] -> Output Line: 2
1
1
0
1
Address: [1, 1, 1, 0] -> Output Line: 1
1
1
1
0
Address: [1, 1, 1, 1] -> Output Line: 0
1
1
1
1
